# 05 — Benchmark

## Setup

In [1]:
import os, sys, pathlib, warnings
root = pathlib.Path.cwd()
if root.name == "notebooks":
    root = root.parent
    os.chdir(root)
sys.path.insert(0, str(root))

# Warnings print the full path of the file that raised them.
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", message=".*IProgress.*")

## The three models

In [2]:
import json, csv, statistics as st
from pathlib import Path
from src.model import build_model, normalization_for, ARCHITECTURES

print(f"{'architecture':26} {'params':>12}   normalization")
for arch in ARCHITECTURES:
    model = build_model({"arch": arch, "base_channels": 32, "depth": 4})
    mean, _ = normalization_for({"arch": arch})
    n = sum(p.numel() for p in model.parameters())
    print(f"{arch:26} {n:>12,}   {'0.5/0.5' if mean[0] == 0.5 else 'ImageNet'}")
    del model

architecture                     params   normalization
unet_scratch                  7,763,041   0.5/0.5
unet_resnet34                24,436,369   ImageNet
deeplabv3plus_resnet34       22,437,457   ImageNet


## Test results

In [3]:
paths = {
    "U-Net from scratch":     ("results/metrics.json", "runs/baseline"),
    "U-Net + ResNet-34":      ("results/benchmark/unet_resnet34/metrics_test.json", "runs/unet_resnet34"),
    "DeepLabV3+ + ResNet-34": ("results/benchmark/deeplabv3plus_resnet34/metrics_test.json", "runs/deeplabv3plus_resnet34"),
}

def best_val(run):
    p = Path(run) / "history.csv"
    return max(float(r["val_iou"]) for r in csv.DictReader(open(p))) if p.exists() else 0.0

print(f"{'model':26} {'val':>7} {'test':>8} {'prec':>8} {'recall':>8}  epoch")
scores = {}
for label, (mp, run) in paths.items():
    if not Path(mp).exists():
        print(f"{label:26}   missing {mp}"); continue
    d = json.load(open(mp)); o = d["overall"]
    scores[label] = o["iou"]
    print(f"{label:26} {best_val(run):>7.4f} {o['iou']:>8.4f} {o['precision']:>8.4f} {o['recall']:>8.4f}  {d['checkpoint_epoch']}")

if len(scores) == 3:
    print(f"\nspread {max(scores.values()) - min(scores.values()):.4f} IoU")

model                          val     test     prec   recall  epoch
U-Net from scratch          0.7450   0.7565   0.8574   0.8654  41
U-Net + ResNet-34           0.7498   0.7643   0.8636   0.8692  50
DeepLabV3+ + ResNet-34      0.7374   0.7545   0.8612   0.8589  50

spread 0.0098 IoU


## Run-to-run variation

In [4]:
print(f"{'run':26} {'mean':>8} {'sd':>8} {'range':>8}   last 10 epochs")
for label, (_, run) in paths.items():
    p = Path(run) / "history.csv"
    if not p.exists(): continue
    v = [float(r["val_iou"]) for r in csv.DictReader(open(p))][-10:]
    print(f"{label:26} {st.mean(v):>8.4f} {st.pstdev(v):>8.4f} {max(v)-min(v):>8.4f}")

run                            mean       sd    range   last 10 epochs
U-Net from scratch           0.7375   0.0037   0.0136
U-Net + ResNet-34            0.7451   0.0038   0.0135
DeepLabV3+ + ResNet-34       0.7343   0.0021   0.0065


## Reproducing a benchmark run

```bash
python -m src.train --arch unet_resnet34 --out runs/unet_resnet34

python -m src.evaluate --checkpoint runs/unet_resnet34/best.pt \
  --predictions runs/unet_resnet34/predictions_test --set test \
  --out results/benchmark/unet_resnet34/metrics_test.json
```